In [1]:

import json
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

# Load data
def load_and_preprocess(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    rows = []
    label_mapping = {"yes": 1, "no": 0, "maybe": 2}

    for pmid, item in data.items():
        question = item["QUESTION"]
        contexts = " ".join(item["CONTEXTS"])

        rows.append({
            "pmid": pmid,
            "text": f"{question} {contexts}",
            "label": label_mapping[item["final_decision"]]
        })

    return pd.DataFrame(rows)


# Per fold evaluation
accuracies = []
precisions, recalls, f1s = [], [], []

# Cross-fold evaluation
all_y_true = []
all_y_pred = []
all_probs = []

errors = []

# Loop over folds 0–9
for fold in range(10):
    train_path = f"data/pqal_fold{fold}/train_set.json"
    dev_path = f"data/pqal_fold{fold}/dev_set.json"

    train_df = load_and_preprocess(train_path)
    dev_df = load_and_preprocess(dev_path)

    # TF-IDF
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(train_df['text'])
    X_dev = vectorizer.transform(dev_df['text'])

    y_train = train_df['label']
    y_dev = dev_df['label']

    # Model
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_dev)
    probs = model.predict_proba(X_dev)

    # Per fold metrics
    acc = accuracy_score(y_dev, predictions)
    accuracies.append(acc)

    p, r, f, _ = precision_recall_fscore_support(
        y_dev, predictions, average='macro', zero_division=0)

    precisions.append(p)
    recalls.append(r)
    f1s.append(f)

    #print(f"Accuracy: {acc:.3f} | F1 macro: {f:.3f}")

    # Aggregated metrics
    all_y_true.extend(y_dev)
    all_y_pred.extend(predictions)
    all_probs.extend(np.max(probs, axis=1))


# Final summary
print("Cross-validation summary:")
print(f"Accuracy: {np.mean(accuracies):.3f} ± {np.std(accuracies):.3f}")
print(f"Precision (macro): {np.mean(precisions):.3f}")
print(f"Recall (macro): {np.mean(recalls):.3f}")
print(f"F1 (macro): {np.mean(f1s):.3f}")

# Classification report
print("Classification report (across all folds):")
print(classification_report(
    all_y_true,
    all_y_pred,
    target_names=["no", "yes", "maybe"],
    digits=3,
    zero_division=0))

# Confusion matrix
print("Confusion matrix:")
conf_matrix = confusion_matrix(all_y_true, all_y_pred)
print(conf_matrix)

print("Confusion matrix (normalised):") # as percentages
conf_matrix_norm = conf_matrix.astype(float) / conf_matrix.sum(axis=1, keepdims=True)
print(np.round(conf_matrix_norm, 3))

# Confidence analysis
all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
all_probs = np.array(all_probs)

correct = all_y_true == all_y_pred

print("Confidence Analysis:")
print(f"Avg confidence (correct): {all_probs[correct].mean():.3f}")
print(f"Avg confidence (wrong):   {all_probs[~correct].mean():.3f}")

Cross-validation summary:
Accuracy: 0.516 ± 0.065
Precision (macro): 0.371
Recall (macro): 0.366
F1 (macro): 0.353
Classification report (across all folds):
              precision    recall  f1-score   support

          no      0.377     0.308     0.339       169
         yes      0.572     0.736     0.643       276
       maybe      0.429     0.055     0.097        55

    accuracy                          0.516       500
   macro avg      0.459     0.366     0.360       500
weighted avg      0.490     0.516     0.480       500

Confusion matrix:
[[ 52 115   2]
 [ 71 203   2]
 [ 15  37   3]]
Confusion matrix (normalised):
[[0.308 0.68  0.012]
 [0.257 0.736 0.007]
 [0.273 0.673 0.055]]
Confidence Analysis:
Avg confidence (correct): 0.419
Avg confidence (wrong):   0.416


# Test Set

In [2]:
# Merge training with dev data
all_train_dfs = []

for fold in range(10):
    train_path = f"data/pqal_fold{fold}/train_set.json"
    dev_path = f"data/pqal_fold{fold}/dev_set.json"

    train_df = load_and_preprocess(train_path)
    dev_df = load_and_preprocess(dev_path)

    # Combine both — since across folds, dev sets are just held-out splits
    all_train_dfs.append(train_df)
    all_train_dfs.append(dev_df)

# Merge everything
full_train_df = pd.concat(all_train_dfs, ignore_index=True)

# Prepare test data
test_path = f"data/test_set.json"
test_df = load_and_preprocess(test_path)

 # TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_full_train = vectorizer.fit_transform(full_train_df['text'])
X_test = vectorizer.transform(test_df['text'])

y_full_train = full_train_df['label']
y_test = test_df['label']

# Model
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_full_train, y_full_train)

# Predict
preds_test = model.predict(X_test)
probs_test = model.predict_proba(X_test)

# Per fold metrics
acc = accuracy_score(y_test, preds_test)

p, r, f, _ = precision_recall_fscore_support(
    y_test, preds_test, average='macro', zero_division=0)

print(f"Accuracy: {acc:.3f} | F1 macro: {f:.3f}")

Accuracy: 0.514 | F1 macro: 0.309


# CSV export

In [3]:
# Data frame
df_q3 = pd.DataFrame({"pubmed_id":test_df['pmid'], "y_true":y_test, "y_pred":preds_test})
df_q3['correct'] = (df_q3['y_pred'] == df_q3['y_true']).astype(int)
df_q3.to_csv("exp0.csv", index=False)